# 00 — Environment + data setup (Colab)

Run this once per Colab session to (a) install the `region_grounded` package and (b) stage a CC3M subset.
Expected runtime: A100 spot (B/16) or L4 (B/32 fallback).

In [ ]:
# 1) Mount Drive for persistent outputs/checkpoints (recommended)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Clone the (private) repo and install in editable mode.
#
# The repo is private, so Colab needs auth. Easiest path: paste a fine-scoped
# Personal Access Token (PAT) below; we only ever use it for this one git clone.
# Create one at https://github.com/settings/tokens (scope: 'repo' only).
import os, getpass, subprocess
REPO_OWNER = 'Ani0202'
REPO_NAME  = 'region-grounded'
REPO_DIR   = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    token = os.environ.get('GH_TOKEN') or getpass.getpass('GitHub PAT (input hidden): ')
    url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    subprocess.check_call(['git', 'clone', url, REPO_DIR])
    del token, url   # don't leave the token sitting in a Python variable
%cd $REPO_DIR
!pip -q install -e .

## Data — stream a CC3M subset from `pixparse/cc3m-wds`

We use the HuggingFace webdataset mirror, which serves pre-fetched image
shards — no URL-fetching, no Google CC3M registration. Pulls roughly
50k 224-px images in 20–40 min on Colab's network.

If you'd rather use the official TSV via `img2dataset`, pass
`--source tsv --tsv <path>` instead.

In [ ]:
# Persist images under Drive so we don't redownload on each session restart
DATA_DIR = '/content/drive/MyDrive/region-grounded/data/cc3m'
SUBSET = 50000
!python scripts/download_cc3m.py --out $DATA_DIR --subset $SUBSET
!ln -sfn $DATA_DIR data/cc3m
!ls data/cc3m | head

In [ ]:
# 3) Smoke test — load SigLIP via the SCLIP encoder and confirm GPU is hot
import torch
from region_grounded import load_config
from region_grounded.stage1_extract import SCLIPVisionEncoder

cfg = load_config('configs/default.yaml')
enc = SCLIPVisionEncoder(cfg.stage1.siglip_model, dtype=torch.float16)
print('grid =', enc.grid, ' dim =', enc.dim, ' device =', enc.device)
assert enc.device.type == 'cuda', 'No GPU detected — switch runtime to A100/L4'

In [ ]:
# 4) Optional — run the smoke test (~10s on CPU) to catch wiring regressions before Stage 1
!python tests/smoke.py